# Data Driven Assignment 2

Dette er en enkel notebook for prosjektet.

In [1]:
import pandas as pd

print("Notebook klar for analyse og visualisering.")

Notebook klar for analyse og visualisering.


In [2]:
%pip install opencv-python scikit-image
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
%pip install "opencv-python<5"

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Found existing installation: opencv-python 4.14.0.94
Uninstalling opencv-python-4.14.0.94:
  Successfully uninstalled opencv-python-4.14.0.94
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-4.14.0.94-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
Using cached opencv_python-4.14.0.94-cp37-abi3-macosx_13_0_arm64.whl (46.5 MB)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
print(cv2.__version__)
print(hasattr(cv2, "CascadeClassifier"))

4.14.0
True


In [4]:
from pathlib import Path
import numpy as np
import cv2

MAPPE = Path("MAPPE = Path(facial-emotion-recognition/images)")        # endre til din egen bane
UTTRYKK = ["Anger", "Contempt", "Disgust", "Fear",
           "Happy", "Neutral", "Sad", "Surprised"]
PERSONER = range(19)
FRO = 42                      # fast frø for alt som er tilfeldig

def finn_fil(person, uttrykk):
    for fil in (MAPPE / str(person)).iterdir():
        if fil.stem.lower() == uttrykk.lower():
            return fil
    raise FileNotFoundError(f"Fant ikke {uttrykk} for person {person}")

def les_graa(person, uttrykk, langside=512):
    """Leser ett bilde som gråtone, skalert slik at lengste side = langside.
    Returnerer verdier 0-255 (uint8)."""
    data = np.fromfile(finn_fil(person, uttrykk), dtype=np.uint8)
    bilde = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    skala = langside / max(bilde.shape)
    return cv2.resize(bilde, None, fx=skala, fy=skala,
                      interpolation=cv2.INTER_AREA)

def til_01(bilde):
    """Normaliserer fra 0-255 til [0, 1]."""
    return bilde.astype(np.float64) / 255.0

_detektor = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def ansiktsutsnitt(bilde, storrelse=64):
    """Klipper ut ansiktet og skalerer til storrelse x storrelse.
    Returnerer (utsnitt, funnet). Finner den ikke noe ansikt,
    brukes et kvadrat midt i bildet og funnet = False."""
    minste = min(bilde.shape) // 5
    treff = _detektor.detectMultiScale(bilde, scaleFactor=1.1,
                                       minNeighbors=5,
                                       minSize=(minste, minste))
    if len(treff) > 0:
        x, y, b, h = max(treff, key=lambda t: t[2] * t[3])
        funnet = True
    else:
        b = h = min(bilde.shape)
        x = (bilde.shape[1] - b) // 2
        y = (bilde.shape[0] - h) // 2
        funnet = False
    utsnitt = bilde[y:y + h, x:x + b]
    utsnitt = cv2.resize(utsnitt, (storrelse, storrelse),
                         interpolation=cv2.INTER_AREA)
    return utsnitt, funnet

In [5]:
from felles import *
import pandas as pd

ansikter, rader = [], []
for person in PERSONER:
    for uttrykk in UTTRYKK:
        if (person, uttrykk) in UTELATES:
            continue
        utsnitt, funnet = ansiktsutsnitt(les_graa(person, uttrykk),
                                         bruk_detektor=person not in MIDTUTSNITT)
        ansikter.append(utsnitt)
        rader.append({"person": person, "uttrykk": uttrykk, "funnet": funnet})

ansikter = np.array(ansikter)
etiketter = pd.DataFrame(rader)
np.save("ansikter_64.npy", ansikter)
etiketter.to_csv("ansikter_64.csv", index=False)

print("Form:", ansikter.shape)
print("Midtutsnitt brukt i", (~etiketter["funnet"]).sum(), "bilder")

ruter = {(r["person"], r["uttrykk"]): a for r, a in zip(rader, ansikter)}
hvit = np.full((64, 64), 255, dtype=np.uint8)
rader_bilde = [np.hstack([ruter.get((p, u), hvit) for u in UTTRYKK]) for p in PERSONER]
cv2.imwrite("ansikter_oversikt.png", np.vstack(rader_bilde))

Form: (149, 64, 64)
Midtutsnitt brukt i 8 bilder


True

In [6]:
from felles-2 import *
print(MAPPE, MAPPE.exists())

SyntaxError: invalid syntax (365240955.py, line 1)

In [ ]:
from felles import *

sma = []
for person in PERSONER:
    for uttrykk in UTTRYKK:
        bilde = les_graa(person, uttrykk, langside=200)
        lerret = np.full((200, 200), 255, dtype=np.uint8)
        h, b = bilde.shape
        lerret[(200 - h) // 2:(200 - h) // 2 + h,
               (200 - b) // 2:(200 - b) // 2 + b] = bilde
        sma.append(lerret)

rader_bilde = [np.hstack(sma[i:i + 8]) for i in range(0, len(sma), 8)]
cv2.imwrite("oversikt.jpg", np.vstack(rader_bilde))

True